# AlphaDent YOLO26 4-Class Segmentation Pipeline (End-to-End)

This notebook contains the complete pipeline for downloading the AlphaDent dataset, converting it to a 4-class problem (Abrasion, Filling, Crown, Caries), and training, validating, and performing inference using **YOLO26 segment models**.

### Models Supported:
- `yolo26n-seg.pt` (Nano)
- `yolo26s-seg.pt` (Small)
- `yolo26m-seg.pt` (Medium)
- `yolo26l-seg.pt` (Large)
- `yolo26x-seg.pt` (Extra Large)

Developed for running on **Kaggle**.

## 1. Clone Repository & Setup Working Directory

In [ ]:
# Clone repo if not running inside it already
import os
if not os.path.exists('/kaggle/working/AlphaDentYolov26'):
    print("Cloning AlphaDentYolov26 repository...")
    !git clone https://github.com/ShMazumder/AlphaDentYolov26 /kaggle/working/AlphaDentYolov26/
else:
    print("Repository already exists.")

# Change directory to the repository root
os.chdir('/kaggle/working/AlphaDentYolov26/')
print(f"Current working directory: {os.getcwd()}")

# Install dependencies
print("Installing dependencies...")
!pip install -q -r requirements.txt
!pip install -q matplotlib opencv-python

## 2. Configuration

Configure model selection and training hyperparameters here.

In [ ]:
import torch

# Experiment Grid Configuration
GRID_MODELS = ['yolo26m-seg.pt', 'yolo26l-seg.pt']
GRID_IMAGE_SIZES = [640, 960]
TARGET_BATCH_SIZE = 16  # Initial target batch size; will auto-fallback to 8 or 4 on CUDA OOM
EPOCHS = 100

# Override Checkpoint Controls
# If False, will skip running if results/weights for the configuration already exist on disk
OVERRIDE_TRAIN = False
OVERRIDE_VAL = False
OVERRIDE_XAI = False

# Dataset configuration file path (SEPARATE non-destructive 4-class dataset)
DATA_CONFIG = './AlphaDent_4class/yolo_seg_train_4class.yaml'
EXPECTED_NC = 4                       # guard against loading a wrong-nc (e.g. 9-class) checkpoint
RUNS_ROOT   = 'runs_4classes/segment' # isolate 4-class runs from the 9-class runs/segment

# Device configuration: list of all available GPU IDs, or 'cpu'
DEVICE = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else 'cpu'

# Validation & Inference Settings
CONF_THRESHOLD = 0.001
IOU_THRESHOLD = 0.5

print("--- Experiment Grid Setup ---")
print(f"Models to test: {GRID_MODELS}")
print(f"Resolutions to test: {GRID_IMAGE_SIZES}")
print(f"Device: {DEVICE}")
print(f"Target Batch Size: {TARGET_BATCH_SIZE}, Epochs per run: {EPOCHS}")
print(f"Override controls -> Train: {OVERRIDE_TRAIN}, Val: {OVERRIDE_VAL}, XAI: {OVERRIDE_XAI}")

## 3. Dataset Download & Unpacking

Download the dataset from Zenodo and extract it to the correct path.

In [ ]:
import zipfile
import shutil
import os

input_dir = "/kaggle/input/competitions/alpha-dent/AlphaDent"
working_dir = "/kaggle/working/AlphaDentYolov26/AlphaDent"
zip_path = "/kaggle/working/dataset.zip"

# Check if Kaggle input contains the dataset
if os.path.exists(os.path.join(input_dir, 'images')) and os.listdir(os.path.join(input_dir, 'images')):
    # print("Dataset found in Kaggle input. Preparing files...")
    # os.makedirs(working_dir, exist_ok=True)
    # # 1. Symlink the large images folder (read-only is fine)
    # src_images = os.path.join(input_dir, 'images')
    # dst_images = os.path.join(working_dir, 'images')
    # if not os.path.exists(dst_images):
    #     os.symlink(src_images, dst_images)
    #     print("Symlinked images directory.")
    # # 2. Copy the small labels folder (needs to be writeable for YOLO cache writing)
    # src_labels = os.path.join(input_dir, 'labels')
    # dst_labels = os.path.join(working_dir, 'labels')
    # if not os.path.exists(dst_labels):
    #     print("Copying labels directory to allow cache writing...")
    #     shutil.copytree(src_labels, dst_labels)
    #     print("Copied labels directory.")
    # print("Dataset prepared successfully using Kaggle input.")

    print("Dataset found in Kaggle input. Copying dataset...")
    shutil.copytree(input_dir, working_dir, dirs_exist_ok=True)
    print("Dataset copied successfully. Using input dataset.")
else:
    # Fallback to downloading from Zenodo
    images_dir = os.path.join(working_dir, 'images')
    if not os.path.exists(images_dir) or not os.listdir(images_dir):
        print("Downloading AlphaDent dataset from Zenodo...")
        !wget -q --show-progress "https://zenodo.org/records/16582489/files/AlphaDent.zip?download=1" -O {zip_path}
        
        print("Extracting dataset...")
        os.makedirs(working_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(working_dir)
        
        # Remove zip to save disk space
        if os.path.exists(zip_path):
            os.remove(zip_path)
            
        # Check if nested folder exists after extraction and move files up if needed
        nested_dir = os.path.join(working_dir, 'AlphaDent')
        if os.path.exists(nested_dir) and os.path.isdir(nested_dir):
            print("Adjusting dataset directory structure...")
            for item in os.listdir(nested_dir):
                shutil.move(os.path.join(nested_dir, item), working_dir)
            os.rmdir(nested_dir)
            
        print("Dataset prepared successfully in working directory.")
    else:
        print("Dataset already exists in working directory.")
        
# --- Build a SEPARATE 4-class dataset (NON-DESTRUCTIVE: 9-class labels stay intact) ---
def build_4class_dataset(src_root, dst_root):
    for split in ['train', 'valid']:
        src_img = os.path.join(src_root, 'images', split)
        src_lbl = os.path.join(src_root, 'labels', split)
        dst_img = os.path.join(dst_root, 'images', split)
        dst_lbl = os.path.join(dst_root, 'labels', split)
        os.makedirs(dst_img, exist_ok=True)
        os.makedirs(dst_lbl, exist_ok=True)
        # link images (avoid duplicating the large image set)
        if os.path.isdir(src_img):
            for fn in os.listdir(src_img):
                d = os.path.join(dst_img, fn)
                if not os.path.exists(d):
                    try:
                        os.symlink(os.path.abspath(os.path.join(src_img, fn)), d)
                    except OSError:
                        shutil.copy2(os.path.join(src_img, fn), d)
        # write CONVERTED label COPIES (class>=3 -> 3); originals untouched
        n = 0
        if os.path.isdir(src_lbl):
            for fn in os.listdir(src_lbl):
                if not fn.endswith('.txt'):
                    continue
                out = []
                with open(os.path.join(src_lbl, fn)) as f:
                    for line in f:
                        parts = line.split()
                        if parts:
                            if int(parts[0]) >= 3:
                                parts[0] = '3'
                            out.append(' '.join(parts) + '\n')
                with open(os.path.join(dst_lbl, fn), 'w') as f:
                    f.writelines(out)
                n += 1
        print(f"  {split}: linked images, wrote {n} 4-class label files")
    return dst_root

dst_root = working_dir + '_4class'   # e.g. .../AlphaDent_4class  (sibling of the 9-class AlphaDent)
print("Building non-destructive 4-class dataset...")
build_4class_dataset(working_dir, dst_root)

# Remove any writable YOLO label cache in the new dataset so it re-scans the 4-class labels
for _c in [os.path.join(dst_root, 'labels', 'train.cache'), os.path.join(dst_root, 'labels', 'valid.cache')]:
    if os.path.exists(_c):
        os.remove(_c)

# Generate the 4-class YAML dataset configuration (points at the SEPARATE dataset)
yaml_content = 'path: ./AlphaDent_4class/\n\ntrain: images/train\nval: images/valid\nnames:\n  0: Abrasion\n  1: Filling\n  2: Crown\n  3: Caries\n'
yaml_path = os.path.join(dst_root, 'yolo_seg_train_4class.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)
print(f"Generated 4-class YAML config at: {yaml_path} (original 9-class labels untouched)")


## 3.1 Dataset Statistics

Compute and display summary statistics of the dataset images and instance distribution per class.

In [ ]:
import os
import pandas as pd
import yaml
from IPython.display import display, Markdown

working_dir = "./AlphaDent"
train_labels_dir = os.path.join(working_dir, "labels/train")
val_labels_dir = os.path.join(working_dir, "labels/valid")

# Load class configuration dynamically from the dataset YAML file
with open(DATA_CONFIG, 'r') as f:
    data_yaml = yaml.safe_load(f)
class_names = data_yaml['names']

def compute_stats(labels_dir):
    stats = {int(k): 0 for k in class_names.keys()}
    img_count = 0
    if os.path.exists(labels_dir):
        for filename in os.listdir(labels_dir):
            if filename.endswith(".txt"):
                img_count += 1
                filepath = os.path.join(labels_dir, filename)
                with open(filepath, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) > 0:
                            class_idx = int(parts[0])
                            if class_idx in stats:
                                stats[class_idx] += 1
    return img_count, stats

train_imgs, train_stats = compute_stats(train_labels_dir)
val_imgs, val_stats = compute_stats(val_labels_dir)

# 1. Image Counts Summary Table
img_df = pd.DataFrame({
    "Split": ["Train", "Validation", "Total"],
    "Images": [train_imgs, val_imgs, train_imgs + val_imgs]
})

# 2. Class Instance Distribution Table
class_rows = []
for idx, name in sorted(class_names.items()):
    t_count = train_stats.get(idx, 0)
    v_count = val_stats.get(idx, 0)
    class_rows.append({
        "Class ID": idx,
        "Class Name": name,
        "Train Instances": t_count,
        "Val Instances": v_count,
        "Total Instances": t_count + v_count
    })
class_df = pd.DataFrame(class_rows)

print("### Dataset Image Summary")
display(Markdown(img_df.to_markdown(index=False)))
print("\n### Dataset Instance Distribution by Class")
display(Markdown(class_df.to_markdown(index=False)))

## 4. Train Model

Load the selected YOLO26 model and start training.

In [ ]:
import os
if os.path.exists('/kaggle/working/AlphaDentYolov26'):
    os.chdir('/kaggle/working/AlphaDentYolov26')

import gc
import torch
from ultralytics import YOLO

# Ensure W&B is disabled to run offline smoothly
os.environ['WANDB_DISABLED'] = 'true'

print("Starting grid search training loop...")

for model_name in GRID_MODELS:
    for img_size in GRID_IMAGE_SIZES:
        model_base = model_name.replace('-seg.pt', '').replace('.pt', '')
        
        # Check if weights already exist for any batch size
        import glob
        existing_weights = []
        for b in [TARGET_BATCH_SIZE, 8, 4]:
            p_check = f"runs_4classes/segment/yolo_seg_{model_base}_proj_{img_size}_b{b}_e{EPOCHS}/train/weights/best.pt"
            if os.path.exists(p_check):
                existing_weights.append(p_check)
            p_check_alt1 = f"yolo_seg_{model_base}_proj_{img_size}_b{b}_e{EPOCHS}/train/weights/best.pt"
            if os.path.exists(p_check_alt1):
                existing_weights.append(p_check_alt1)
            p_check_alt2 = f"runs/yolo_seg_{model_base}_proj_{img_size}_b{b}_e{EPOCHS}/train/weights/best.pt"
            if os.path.exists(p_check_alt2):
                existing_weights.append(p_check_alt2)
        
        weights_exist = len(existing_weights) > 0
        existing_weights_path = existing_weights[0] if weights_exist else None
        
        if weights_exist and not OVERRIDE_TRAIN:
            print(f"Skipping training for {model_name} at size {img_size} (weights exist at {existing_weights_path})")
            continue
            
        print(f"\n==================================================")
        print(f"TRAINING: {model_name} | Image Size: {img_size}")
        print(f"==================================================")
        
        current_batch = TARGET_BATCH_SIZE
        trained_successfully = False
        
        while current_batch >= 4:
            project_name = f'yolo_seg_{model_base}_proj_{img_size}_b{current_batch}_e{EPOCHS}'
            print(f"Attempting training with batch size: {current_batch} | Project: {project_name}...")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                
            try:
                model = YOLO(model_name)
                results = model.train(
                    data=DATA_CONFIG,
                    epochs=EPOCHS,
                    imgsz=img_size,
                    batch=current_batch,
                    project=f"{RUNS_ROOT}/{project_name}",
                    name="train",
                    exist_ok=True,  # Reuse/overwrite output directory to avoid duplicate folders
                    deterministic=False,
                    plots=True,
                    device=DEVICE,
                )
                trained_successfully = True
                print(f"Successfully trained {model_name} at size {img_size} with batch {current_batch}!")
                break
            except Exception as e:
                print(f"⚠️ Training failed for {model_name} at size {img_size} with batch {current_batch}: {e}")
                current_batch //= 2
                if current_batch < 4:
                    print(f"❌ Batch size fell below 4. Re-raising the last exception.")
                    raise e
                print(f"Retrying with batch size: {current_batch}...")
                    
print("Training phase finished.")

## 5. Model Validation

Evaluate the trained model's performance on the validation dataset.

In [ ]:
import os
if os.path.exists('/kaggle/working/AlphaDentYolov26'):
    os.chdir('/kaggle/working/AlphaDentYolov26')

import gc
import torch
import pandas as pd
from ultralytics import YOLO

results_file = "grid_search_results.csv"
grid_results = []

# Load previously saved results if file exists
if os.path.exists(results_file):
    try:
        df_existing = pd.read_csv(results_file)
        grid_results = df_existing.to_dict('records')
        print(f"Loaded {len(grid_results)} existing results from {results_file}.")
    except Exception as e:
        print(f"Could not load existing CSV: {e}. Starting fresh.")

print("Starting validation loop...")

for model_name in GRID_MODELS:
    for img_size in GRID_IMAGE_SIZES:
        model_base = model_name.replace('-seg.pt', '').replace('.pt', '')
        
        weights_path = None
        project_name = None
        for b in [TARGET_BATCH_SIZE, 8, 4]:
            p_proj = f'yolo_seg_{model_base}_proj_{img_size}_b{b}_e{EPOCHS}'
            possible_paths = [
                f"runs_4classes/segment/{p_proj}/train/weights/best.pt",
                f"{p_proj}/train/weights/best.pt",
                f"runs/{p_proj}/train/weights/best.pt",
                f"/kaggle/working/AlphaDentYolov26/runs_4classes/segment/{p_proj}/train/weights/best.pt"
            ]
            for p in possible_paths:
                if os.path.exists(p):
                    weights_path = p
                    project_name = p_proj
                    break
            if weights_path:
                break
                
        if not weights_path:
            # Fallback for folder naming if not trained in this session
            project_name = f'yolo_seg_{model_base}_proj_{img_size}_b{TARGET_BATCH_SIZE}_e{EPOCHS}'
            
        # Check if this run is already completed in grid_results
        already_evaluated = False
        for r in grid_results:
            if r.get('model') == model_name and r.get('img_size') == img_size:
                already_evaluated = True
                break
                
        if already_evaluated and not OVERRIDE_VAL:
            print(f"Skipping validation for {model_name} at size {img_size} (metrics exist in CSV)")
            continue
            
        if not weights_path:
            print(f"⚠️ Weights not found for {model_name} at size {img_size}. Skipping validation.")
            continue
            
        print(f"\n==================================================")
        print(f"VALIDATING: {model_name} | Image Size: {img_size}")
        print(f"==================================================")
        
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        val_device = DEVICE[0] if isinstance(DEVICE, list) else DEVICE
        model_val = YOLO(weights_path)
        _nc = getattr(model_val.model, 'nc', None) or len(model_val.names)
        assert _nc == EXPECTED_NC, (f"Loaded checkpoint nc={_nc} but EXPECTED_NC={EXPECTED_NC}. "
                                    f"Wrong (likely 9-class) weights at {weights_path}; refusing to score to avoid contaminated metrics.")
        
        try:
            metrics = model_val.val(
                data=DATA_CONFIG,
                project=f"{RUNS_ROOT}/{project_name}",
                name="val",
                imgsz=img_size,
                batch=1,
                iou=IOU_THRESHOLD,
                conf=CONF_THRESHOLD,
                half=True,
                save_json=False,
                save_txt=False,
                save_conf=False,
                plots=True,
                device=val_device,
            )
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and val_device != "cpu":
                print("⚠️ CUDA Out of Memory during validation. Retrying on CPU...")
                gc.collect()
                torch.cuda.empty_cache()
                metrics = model_val.val(
                    data=DATA_CONFIG,
                    project=f"{RUNS_ROOT}/{project_name}",
                    name="val",
                    imgsz=img_size,
                    batch=1,
                    iou=IOU_THRESHOLD,
                    conf=CONF_THRESHOLD,
                    save_json=False,
                    save_txt=False,
                    save_conf=False,
                    plots=True,
                    device="cpu",
                )
            else:
                raise e
                
        # Extract model parameters count dynamically
        params_count = sum(p.numel() for p in model_val.model.parameters())
        
        result_entry = {
            'model': model_name,
            'img_size': img_size,
            'params_M': round(params_count / 1e6, 2),
            'Box_Precision': metrics.box.mp,
            'Box_Recall': metrics.box.mr,
            'Box_mAP50': metrics.box.map50,
            'Box_mAP50_95': metrics.box.map,
            'Mask_Precision': metrics.seg.mp,
            'Mask_Recall': metrics.seg.mr,
            'Mask_mAP50': metrics.seg.map50,
            'Mask_mAP50_95': metrics.seg.map
        }
        
        # Update or append
        existing_idx = -1
        for idx, r in enumerate(grid_results):
            if r.get('model') == model_name and r.get('img_size') == img_size:
                existing_idx = idx
                break
        if existing_idx != -1:
            grid_results[existing_idx] = result_entry
        else:
            grid_results.append(result_entry)
            
        # Save intermediate CSV
        pd.DataFrame(grid_results).to_csv(results_file, index=False)
        print(f"Saved results to {results_file}.")

# Display Final Table
if grid_results:
    df_final = pd.DataFrame(grid_results)
    print("\n" + "="*50)
    print("GRID SEARCH RESULTS SUMMARY TABLE")
    print("="*50)
    try:
        print(df_final.to_markdown(index=False))
    except Exception:
        print(df_final.to_string(index=False))
else:
    print("No validation results found.")

## 6. Inference and Visualization

Perform inference on test images and display the predictions.

In [ ]:
import os
if os.path.exists('/kaggle/working/AlphaDentYolov26'):
    os.chdir('/kaggle/working/AlphaDentYolov26')

import glob
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Find last successfully trained model weights
sel = None
for model_name in GRID_MODELS:
    for img_size in GRID_IMAGE_SIZES:
        model_base = model_name.replace('-seg.pt','').replace('.pt','')
        project_name = f'yolo_seg_{model_base}_proj_{img_size}_b{TARGET_BATCH_SIZE}_e{EPOCHS}'
        # Check possible weights locations
        possible_paths = [
            f"runs_4classes/segment/{project_name}/train/weights/best.pt",
            f"{project_name}/train/weights/best.pt",
            f"runs/{project_name}/train/weights/best.pt",
            f"/kaggle/working/AlphaDentYolov26/runs_4classes/segment/{project_name}/train/weights/best.pt"
        ]
        for p in possible_paths:
            if os.path.exists(p):
                sel = (p, img_size)
                break

if sel is None:
    print("No weights found; skipping inference.")
else:
    weights_path, IMAGE_SIZE = sel
    model_infer = YOLO(weights_path)
    
    test_images_dir = "./AlphaDent/images/valid/"
    output_dir = "./inference_output/"
    
    print(f"Running inference on images from: {test_images_dir}")
    val_device = DEVICE[0] if isinstance(DEVICE, list) else DEVICE
    results_infer = model_infer.predict(
        source=test_images_dir,
        project=output_dir,
        name='preds',
        imgsz=IMAGE_SIZE,
        conf=0.25,  # higher threshold for visual clarity
        iou=0.6,
        half=True,
        save=True,
        save_txt=True,
        device=val_device
    )

# Plot the first few results
possible_dirs = [
    f"{output_dir}/preds",
    "runs_4classes/segment/inference_output/preds",
    "/kaggle/working/AlphaDentYolov26/runs_4classes/segment/inference_output/preds",
    "/kaggle/working/AlphaDentYolov26/inference_output/preds"
]
saved_images = []
for d in possible_dirs:
    saved_images += glob.glob(f"{d}/*.jpg") + glob.glob(f"{d}/*.png")
if saved_images:
    num_display = min(3, len(saved_images))
    fig, axes = plt.subplots(1, num_display, figsize=(18, 6))
    if num_display == 1:
        axes = [axes]
    for idx, img_path in enumerate(saved_images[:num_display]):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(os.path.basename(img_path))
    plt.tight_layout()
    plt.show()
else:
    print("No output images found to display.")

## 7. Model Explainability (XAI)

To build trust in medical diagnosis applications, we must explain the model's predictions. This section implements **post-hoc Explainable AI (XAI)** methods to visualize which image regions the YOLO26 model pays attention to when detecting tooth pathologies.

### Supported CAM Methods:
1. **Eigen-CAM**: Computes principal components of the activations. Gradient-free and extremely robust.
2. **Grad-CAM**: Visualizes gradients of the class score with respect to intermediate activations.
3. **Grad-CAM++**: An advanced gradient-based method using second-order gradients.
4. **XGrad-CAM**: Scales the gradients by the normalized activations.
5. **HiRes-CAM**: Element-wise multiplies activations with gradients; guarantees faithfulness.
6. **Layer-CAM**: Spatially weights activations by positive gradients (excellent for lower/intermediate layers).
7. **EigenGrad-CAM**: First principal component of (Activations * Gradients).

### Literature References:
- **Reference Paper (Medical YOLO XAI)**: Borah et al., 2024. ["A comprehensive study on Explainable AI using YOLO and post hoc method on medical diagnosis"](https://doi.org/10.1088/1742-6596/2919/1/012045).
- **Grad-CAM**: Selvaraju et al., 2017. ["Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization"](https://arxiv.org/abs/1610.02391).
- **Grad-CAM++**: Chattopadhay et al., 2018. ["Grad-CAM++: Generalized Gradient-based Visual Explanations for Deep Convolutional Networks"](https://arxiv.org/abs/1710.11063).
- **XGrad-CAM**: Fu et al., 2020. ["Axiom-based Grad-CAM: Towards Accurate Visualization and Explanation of CNNs"](https://arxiv.org/abs/2008.02312).
- **HiRes-CAM**: Draelos & Carin, 2020. ["Use HiResCAM instead of Grad-CAM for faithful explanations of convolutional neural networks"](https://arxiv.org/abs/2011.08891).
- **Layer-CAM**: Jiang et al., 2021. ["LayerCAM: Exploring Hierarchical Class Activation Maps for Localization"](https://arxiv.org/abs/2104.14818).
- **Eigen-CAM / EigenGrad-CAM**: Muhammad & Yeasin, 2020. ["Eigen-CAM: Class Activation Map using Principal Components"](https://arxiv.org/abs/2008.00299).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os

class YOLOExplainer:
    """Class Activation Mapping for Ultralytics YOLO26 seg models.
    FIXED: robust detection-grid extraction (handles nested tuple outputs),
    LOCALIZED-peak target (so gradient CAMs stay distinct), and a single
    forward+backward per image via generate_all() (memory-safe)."""
    METHODS = ['eigencam','gradcam','gradcam++','xgradcam','hirescam','layercam','eigengradcam']

    def __init__(self, model, target_layer, method='eigencam'):
        self.model = model
        self.target_layer = target_layer
        self.method = method.lower()
        self.activations = None
        self.gradients = None
        self.forward_hook = target_layer.register_forward_hook(self._fwd)
        # always register backward hook so generate_all() works for any method
        self.backward_hook = target_layer.register_full_backward_hook(self._bwd)

    def _fwd(self, m, i, o):
        self.activations = (o[0] if isinstance(o,(list,tuple)) else o)

    def _bwd(self, m, gi, go):
        self.gradients = (go[0] if isinstance(go,(list,tuple)) else go)

    def remove_hooks(self):
        self.forward_hook.remove()
        if hasattr(self,'backward_hook'):
            self.backward_hook.remove()

    @staticmethod
    def _find_det_grid(out, nc):
        """Descend nested list/tuple/dict; return the raw detection grid as
        (channels, anchors) with channels in {4+nc, 4+nc+32}. Picks the tensor
        with the most anchors (the concatenated multi-scale head)."""
        cands, stack = [], [out]
        while stack:
            o = stack.pop()
            if torch.is_tensor(o):
                t = o
                if t.ndim == 3 and t.shape[0] == 1:
                    t = t[0]
                if t.ndim == 2:
                    r, c = t.shape
                    if r in (4+nc, 4+nc+32):
                        cands.append(t)
                    elif c in (4+nc, 4+nc+32):
                        cands.append(t.T)
            elif isinstance(o, dict):
                stack.extend(o.values())
            elif isinstance(o, (list, tuple)):
                stack.extend(o)
        return max(cands, key=lambda t: t.shape[1]) if cands else None

    @staticmethod
    def _combine(method, act, grad):
        m = method.lower()
        if m == 'eigencam':
            C,H,W = act.shape
            M = act.reshape(C,-1).T; M = M - M.mean(0)
            U,_,_ = np.linalg.svd(M, full_matrices=False); p = U[:,0].reshape(H,W)
        elif m == 'gradcam':
            w = grad.mean(axis=(1,2)); p = np.maximum((w[:,None,None]*act).sum(0),0)
        elif m == 'gradcam++':
            g2,g3 = grad**2, grad**3; sa = act.sum(axis=(1,2))
            alpha = g2/(2*g2 + sa[:,None,None]*g3 + 1e-8)
            w = (alpha*np.maximum(grad,0)).sum(axis=(1,2)); p = np.maximum((w[:,None,None]*act).sum(0),0)
        elif m == 'xgradcam':
            sa = act.sum(axis=(1,2)); w = (grad*act).sum(axis=(1,2))/(sa+1e-8)
            p = np.maximum((w[:,None,None]*act).sum(0),0)
        elif m == 'hirescam':
            p = np.maximum((act*grad).sum(0),0)
        elif m == 'layercam':
            p = np.maximum((act*np.maximum(grad,0)).sum(0),0)
        elif m == 'eigengradcam':
            AG = act*np.maximum(grad,0); C,H,W = AG.shape
            M = AG.reshape(C,-1).T; M = M - M.mean(0)
            U,_,_ = np.linalg.svd(M, full_matrices=False); p = U[:,0].reshape(H,W)
        else:
            raise ValueError(method)
        return (p - p.min())/(p.max() - p.min() + 1e-8)

    def _act_grad(self, input_tensor, class_idx=None):
        """One forward + one localized-peak backward -> (class_idx, act, grad) numpy."""
        dev = input_tensor.device
        self.model.model.to(dev)
        self.model.model.zero_grad(set_to_none=True)
        x = input_tensor.clone().requires_grad_(True)
        try:
            outputs = self.model.model(x)
            nc = getattr(self.model.model,'nc',None) or len(self.model.names)
            det = self._find_det_grid(outputs, nc)
            if det is None:
                raise RuntimeError('could not locate detection grid in model output')
            cls_rows = det[4:4+nc, :]                 # (nc, anchors)
            if class_idx is None:
                class_idx = int(cls_rows.sum(dim=-1).argmax())
            score = cls_rows[class_idx, :].max()      # LOCALIZED peak, not global sum
            score.backward()
            act  = self.activations.detach().float().cpu().numpy()[0]
            grad = self.gradients.detach().float().cpu().numpy()[0]
        finally:
            self.model.model.zero_grad(set_to_none=True)
            del x
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        return class_idx, act, grad

    def generate(self, input_tensor, class_idx=None):
        """Single-method (kept for backward compatibility)."""
        if self.method == 'eigencam':
            with torch.no_grad():
                _ = self.model.model(input_tensor.to(next(self.model.model.parameters()).device))
            act = self.activations.detach().float().cpu().numpy()[0]
            return self._combine('eigencam', act, None)
        _, act, grad = self._act_grad(input_tensor, class_idx)
        return self._combine(self.method, act, grad)

    def generate_all(self, input_tensor, class_idx=None):
        """One forward+backward -> dict of all 7 heatmaps. Memory-safe; use this."""
        cid, act, grad = self._act_grad(input_tensor, class_idx)
        return {m: self._combine(m, act, grad) for m in self.METHODS}, cid


In [ ]:
from ultralytics import YOLO
import os, glob, cv2, numpy as np, torch, gc

print("Starting XAI execution loop...")
test_images = glob.glob("./AlphaDent/images/valid/*.jpg") + glob.glob("./AlphaDent/images/valid/*.png")

if not test_images:
    print("No validation images found to explain.")
else:
    img_path = test_images[0]
    print(f"Using image for XAI: {img_path}")
    target_layer_idx = 22
    methods = ['EigenCAM','GradCAM','GradCAM++','XGradCAM','HiResCAM','LayerCAM','EigenGradCAM']

    for model_name in GRID_MODELS:
        for img_size in GRID_IMAGE_SIZES:
            model_base = model_name.replace('-seg.pt','').replace('.pt','')
            weights_path, project_name = None, None
            for b in [TARGET_BATCH_SIZE, 8, 4]:
                p_proj = f'yolo_seg_{model_base}_proj_{img_size}_b{b}_e{EPOCHS}'
                for p in [f"runs_4classes/segment/{p_proj}/train/weights/best.pt",
                          f"{p_proj}/train/weights/best.pt",
                          f"runs/{p_proj}/train/weights/best.pt",
                          f"/kaggle/working/AlphaDentYolov26/runs_4classes/segment/{p_proj}/train/weights/best.pt"]:
                    if os.path.exists(p):
                        weights_path, project_name = p, p_proj; break
                if weights_path: break
            if not weights_path:
                project_name = f'yolo_seg_{model_base}_proj_{img_size}_b{TARGET_BATCH_SIZE}_e{EPOCHS}'

            xai_dir = f"runs_4classes/segment/{project_name}/xai"
            if not OVERRIDE_XAI and all(os.path.exists(os.path.join(xai_dir, f"{m}.jpg")) for m in methods):
                print(f"Skipping XAI for {model_name} at size {img_size} (heatmaps exist in {xai_dir})"); continue
            if not weights_path:
                print(f"WARNING Weights not found for {model_name} at size {img_size}. Skipping XAI."); continue

            print(f"\n==================================================")
            print(f"GENERATING XAI: {model_name} | Image Size: {img_size}")
            print(f"==================================================")

            model_explain = YOLO(weights_path)
            model_explain.model.eval()
            _nc = getattr(model_explain.model, 'nc', None) or len(model_explain.names)
            assert _nc == EXPECTED_NC, f"XAI: checkpoint nc={_nc} != EXPECTED_NC={EXPECTED_NC} ({weights_path})."
            target_layer = model_explain.model.model[target_layer_idx]

            img = cv2.imread(img_path)
            img_resized = cv2.resize(img, (img_size, img_size))
            img_norm = img_resized.astype(np.float32) / 255.0
            input_tensor = torch.from_numpy(img_norm.transpose(2,0,1)).unsqueeze(0)
            val_device = DEVICE[0] if isinstance(DEVICE, list) else DEVICE
            input_tensor = input_tensor.to(val_device)

            explainer = YOLOExplainer(model_explain, target_layer, method='eigencam')
            try:
                heatmaps, cid = explainer.generate_all(input_tensor)   # ONE fwd+bwd -> all 7
                print(f"  target class idx={cid}")
                for method in methods:
                    save_path = os.path.join(xai_dir, f"{method}.jpg")
                    if os.path.exists(save_path) and not OVERRIDE_XAI:
                        print(f"  - Skipping {method} (file exists)"); continue
                    print(f"  - Saving {method}...")
                    show_heatmap(img_path, heatmaps[method.lower()], title=f'{model_name} {method} Map', save_path=save_path)
            except Exception as e:
                print(f"  FAILED XAI for {model_name} @ {img_size}: {e}")
            finally:
                explainer.remove_hooks()
                del explainer, model_explain, input_tensor
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()


## 8. Export Results as ZIP

Compress all training runs, weights, validation plots, XAI heatmaps, and metrics into a single ZIP archive for easy download from Kaggle.

In [ ]:
import os
if os.path.exists('/kaggle/working/AlphaDentYolov26'):
    os.chdir('/kaggle/working/AlphaDentYolov26')

import zipfile
import fnmatch

zip_output = "/kaggle/working/experiment_results.zip"
csv_path = "grid_search_results.csv"
runs_dir = "runs"

exclude_patterns = ["*last.pt", "train_batch*"]

print(f"Creating ZIP archive at {zip_output}...")
with zipfile.ZipFile(zip_output, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # 1. Add CSV file if it exists
    if os.path.exists(csv_path):
        print(f"Adding {csv_path} to zip...")
        zipf.write(csv_path, arcname=csv_path)
    
    # 2. Add files from runs directory
    if os.path.exists(runs_dir):
        for root, dirs, files in os.walk(runs_dir):
            for file in files:
                file_path = os.path.join(root, file)
                
                # Check if file matches any exclude pattern
                exclude = False
                for pattern in exclude_patterns:
                    if fnmatch.fnmatch(file, pattern) or fnmatch.fnmatch(os.path.basename(file_path), pattern):
                        exclude = True
                        break
                
                if not exclude:
                    # Write to zip, keeping relative path under runs/
                    arcname = os.path.relpath(file_path, start=os.path.dirname(runs_dir))
                    zipf.write(file_path, arcname=arcname)
                    
print("ZIP archive created successfully! You can now download 'experiment_results.zip' from Kaggle.")